# 04. Attention and positional representations

![Attention block](../images/04_attention_and_positions.svg)

**Learning goals:** implement stable softmax and scaled attention, enforce masks, split and join heads, construct sinusoidal positions, and inspect residual, LayerNorm, and GELU behavior.

In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn.functional as F

SEED = 23
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
print(f'torch={torch.__version__}, device=cpu')

## 1. Stable softmax

Softmax turns one score row into positive weights that sum to one. Subtracting the row maximum does not change the result, but prevents very large exponentials. `torch.softmax` already uses a stable implementation.

In [ ]:
def stable_softmax(logits, dim=-1):
    shifted = logits - logits.amax(dim=dim, keepdim=True)
    exponents = shifted.exp()
    return exponents / exponents.sum(dim=dim, keepdim=True)

logits = torch.tensor([[1000.0, 1001.0, 999.0]])
weights = stable_softmax(logits)
torch.testing.assert_close(weights, torch.softmax(logits, dim=-1))
torch.testing.assert_close(weights.sum(dim=-1), torch.ones(1))
assert torch.isfinite(weights).all()
print('stable weights:', weights.round(decimals=4))

## 2. Scaled dot-product attention

For `Q,K` shaped `(B,N,d_k)`, `Q @ K.transpose(-2,-1)` produces `(B,N,N)` scores. Dividing by $\sqrt{d_k}$ keeps their typical scale controlled. Each softmax row belongs to one query and sums over keys.

In [ ]:
def scaled_attention(q, k, v, allowed=None):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    if allowed is not None:
        assert allowed.any(dim=-1).all(), 'every query needs at least one key'
        scores = scores.masked_fill(~allowed, float('-inf'))
    attention = torch.softmax(scores, dim=-1)
    return attention @ v, attention

B, N, d_k, d_v = 2, 5, 4, 6
q, k, v = torch.randn(B, N, d_k), torch.randn(B, N, d_k), torch.randn(B, N, d_v)
output, attention = scaled_attention(q, k, v)
assert output.shape == (B, N, d_v) and attention.shape == (B, N, N)
torch.testing.assert_close(attention.sum(-1), torch.ones(B, N))
print('output:', tuple(output.shape), 'attention:', tuple(attention.shape))

## 3. Causal masks

A lower-triangular causal mask lets position $i$ attend only to keys at positions $j\le i$. Apply masks to logits before softmax. A fully forbidden row is invalid because there is no probability distribution to normalize.

In [ ]:
causal = torch.ones(N, N, dtype=torch.bool).tril().expand(B, -1, -1)
causal_output, causal_attention = scaled_attention(q, k, v, causal)
forbidden_values = causal_attention.masked_select(~causal)
torch.testing.assert_close(forbidden_values, torch.zeros_like(forbidden_values))
assert torch.isfinite(causal_output).all()
print('first query weights:', causal_attention[0, 0])

## 4. Split features into multiple heads

With model width `D` and `H` heads, each head has width `D/H`. Reshape `(B,N,D)` to `(B,N,H,d_h)`, transpose to `(B,H,N,d_h)`, attend in parallel, then reverse the steps. `reshape` is safer than `view` after a transpose because storage may be non-contiguous.

In [ ]:
B, N, D, heads = 2, 5, 12, 3
assert D % heads == 0
head_width = D // heads
x = torch.randn(B, N, D)
projection = torch.nn.Linear(D, 3 * D, bias=False)
qkv = projection(x).reshape(B, N, 3, heads, head_width).permute(2, 0, 3, 1, 4)
q_h, k_h, v_h = qkv.unbind(dim=0)
head_out, head_weights = scaled_attention(q_h, k_h, v_h)
joined = head_out.transpose(1, 2).reshape(B, N, D)
assert q_h.shape == (B, heads, N, head_width)
assert head_weights.shape == (B, heads, N, N) and joined.shape == x.shape
print('head scores:', tuple(head_weights.shape), 'joined:', tuple(joined.shape))

## 5. Sinusoidal positions

Attention by itself has no absolute order. Sine and cosine waves provide deterministic position vectors at multiple frequencies. Position 0 alternates 0 and 1, and every position has the model width.

In [ ]:
def sinusoidal_positions(length, width):
    positions = torch.arange(length, dtype=torch.float32).unsqueeze(1)
    even_dims = torch.arange(0, width, 2, dtype=torch.float32)
    rates = torch.exp(-math.log(10_000.0) * even_dims / width)
    angles = positions * rates.unsqueeze(0)
    encoding = torch.zeros(length, width)
    encoding[:, 0::2] = torch.sin(angles)
    encoding[:, 1::2] = torch.cos(angles[:, :encoding[:, 1::2].shape[1]])
    return encoding

position = sinusoidal_positions(N, D)
assert position.shape == (N, D)
torch.testing.assert_close(position[0, 0::2], torch.zeros(D // 2))
torch.testing.assert_close(position[0, 1::2], torch.ones(D // 2))
x_with_position = x + position.unsqueeze(0)
print('position shape:', tuple(position.shape))

## 6. LayerNorm, GELU, and residual paths

LayerNorm centers and scales each token across its feature axis, then applies learned scale and shift. GELU smoothly gates a feed-forward expansion. Residual addition requires identical shapes and gives information and gradients a direct path.

In [ ]:
norm = torch.nn.LayerNorm(D, elementwise_affine=False)
normalized = norm(x_with_position)
torch.testing.assert_close(normalized.mean(-1), torch.zeros(B, N), atol=2e-6, rtol=0)
torch.testing.assert_close(normalized.var(-1, unbiased=False), torch.ones(B, N), atol=3e-5, rtol=0)
ffn = torch.nn.Sequential(torch.nn.Linear(D, 4*D), torch.nn.GELU(), torch.nn.Linear(4*D, D))
residual_result = x_with_position + ffn(normalized)
assert residual_result.shape == x.shape
print('GELU samples:', F.gelu(torch.tensor([-2., 0., 2.])).round(decimals=3))

## 7. A complete pre-norm block

`nn.MultiheadAttention(..., batch_first=True)` keeps `(B,N,D)`. Setting `need_weights=False` avoids returning a potentially large `(N,N)` tensor when it is not needed and lets PyTorch choose optimized attention paths.

In [ ]:
class TinyBlock(torch.nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        self.norm1 = torch.nn.LayerNorm(width)
        self.attn = torch.nn.MultiheadAttention(width, heads, dropout=0.0, batch_first=True)
        self.norm2 = torch.nn.LayerNorm(width)
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(width, 4*width), torch.nn.GELU(), torch.nn.Linear(4*width, width))
    def forward(self, tokens):
        n = self.norm1(tokens)
        tokens = tokens + self.attn(n, n, n, need_weights=False)[0]
        return tokens + self.ffn(self.norm2(tokens))

block = TinyBlock(D, heads).eval()
with torch.no_grad():
    transformed = block(x_with_position)
assert transformed.shape == (B, N, D)
assert torch.isfinite(transformed).all()
print('block output:', tuple(transformed.shape))

## Exercises and final takeaways

**Exercises:** (1) Remove the $\sqrt{d_k}$ scaling and compare attention entropy as `d_k` grows. (2) Build a padding mask and assert every forbidden weight is zero. (3) Permute tokens with and without position vectors and observe attention's permutation behavior.

**Takeaways:** attention is a row-wise probability-weighted value lookup; scaling controls logits; masks define legal information flow; heads create parallel relation spaces; positions supply order; normalization, GELU, and residual paths make a trainable deep block.

## Continue learning

[Previous notebook: 03](03_hierarchical_observations.ipynb) | [Lecture](../lectures/04_attention_and_positions.md) | [Curriculum](../README.md) | [Next notebook: 05](05_masked_latent_prediction.ipynb)